In [ ]:
# raw data
#    ↓
# raw text extraction
#    ↓
# Text cleaning and normalization
#    ↓
# Data creation
#    ↓
# Hugging Face Dataset Conversion
#    ↓
# Tokenization
#    ↓
# LoRA/QLoRA fine-tuning
#    ↓
# Validation loss
#    ↓
# Adapter saving and reloading
#    ↓
# Text continuation inference

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

# If current folder is notebooks, move one level up to project root.
if not (project_root / "utils").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.config import Config
from utils.logger import get_logger

config = Config()
logger = get_logger("non_instruction")
logger.info("Step 1 started")

2026-07-11 20:45:24,413 | INFO | non_instruction | Step 1 started


In [2]:
from pathlib import Path
import pandas as pd
from utils.logger import get_logger

logger = get_logger("non_instruction")

project_root = Path.cwd().resolve()
while not ((project_root / "utils").exists() and (project_root / "data").exists()):
    if project_root.parent == project_root:
        raise RuntimeError("Project root not found")
    project_root = project_root.parent

raw_dir = project_root / Path(config.raw_data_dir)
non_instruction_dir = project_root / Path(config.non_instruction_dir)
non_instruction_dir.mkdir(parents=True, exist_ok=True)

csv_path = raw_dir / config.preferred_raw_csv_filename

# Fallback if filename changed
if not csv_path.exists():
    matches = sorted(raw_dir.glob(config.raw_csv_pattern))
    if not matches:
        raise FileNotFoundError(f"No complaints CSV found in {raw_dir}")
    csv_path = matches[-1]

out_path = non_instruction_dir / config.raw_text_filename

print("CWD:", Path.cwd())
print("Project root:", project_root)
print("Using CSV:", csv_path)
print("Exists:", csv_path.exists())

df = pd.read_csv(csv_path, dtype=str, low_memory=False)

text_col = config.source_text_column
if text_col not in df.columns:
    raise ValueError(f"Column not found: {text_col}")

texts = (
    df[text_col]
    .dropna()
    .astype(str)
    .str.strip()
 )
texts = texts[texts.str.len() > config.min_chars_per_paragraph]

out_path.write_text("\n\n".join(texts.tolist()), encoding="utf-8")

logger.info("Raw extraction complete")
logger.info("Input rows: %s", len(df))
logger.info("Extracted text rows: %s", len(texts))
logger.info("Saved file: %s", out_path.resolve())

print("Extracted rows:", len(texts))
print("Saved to:", out_path.resolve())

CWD: d:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\notebooks
Project root: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT
Using CSV: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\raw\complaints-2026-07-04_02_07.csv
Exists: True


2026-07-11 20:45:29,537 | INFO | non_instruction | Raw extraction complete
2026-07-11 20:45:29,539 | INFO | non_instruction | Input rows: 20859
2026-07-11 20:45:29,540 | INFO | non_instruction | Extracted text rows: 4702
2026-07-11 20:45:29,541 | INFO | non_instruction | Saved file: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\non_instruction_dataset\raw_extracted_text.txt


Extracted rows: 4702
Saved to: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\non_instruction_dataset\raw_extracted_text.txt


In [3]:
# ============================================================
# Text cleaning and final non-instruction data creation
# ============================================================

import re

non_instruction_dir = project_root / Path(config.non_instruction_dir)
non_instruction_dir.mkdir(parents=True, exist_ok=True)
in_path = non_instruction_dir / config.raw_text_filename
out_path = non_instruction_dir / config.non_instruction_filename

raw_text = in_path.read_text(encoding="utf-8")

def clean_text(text: str) -> str:
    # Remove placeholder masks like XXXX
    text = re.sub(r"\bX{2,}\b", " ", text)

    # Normalize URLs/emails/numbers
    text = re.sub(r"https?://\S+|www\.\S+", " [URL] ", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " [EMAIL] ", text)
    text = re.sub(r"\b\d{5,}\b", " [NUMBER] ", text)

    # Keep basic punctuation and normalize spaces
    text = re.sub(r"[^\w\s\.,!?;:'\"()\-\/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
cleaned = [clean_text(p) for p in paragraphs]
cleaned = [p for p in cleaned if len(p) >= config.min_chars_per_paragraph]

out_path.write_text("\n\n".join(cleaned), encoding="utf-8")

logger.info("Cleaning complete")
logger.info("Input paragraphs: %s", len(paragraphs))
logger.info("Final non-instruction paragraphs: %s", len(cleaned))
logger.info("Saved final file: %s", out_path.resolve())

print("Input paragraphs:", len(paragraphs))
print("Final non-instruction paragraphs:", len(cleaned))
print("Saved to:", out_path.resolve())
print("\nSample cleaned text:\n", cleaned[0][:500] if cleaned else "No cleaned text")

2026-07-11 20:45:39,884 | INFO | non_instruction | Cleaning complete
2026-07-11 20:45:39,885 | INFO | non_instruction | Input paragraphs: 18824
2026-07-11 20:45:39,886 | INFO | non_instruction | Final non-instruction paragraphs: 17479
2026-07-11 20:45:39,887 | INFO | non_instruction | Saved final file: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\non_instruction_dataset\non_instruction_data.txt


Input paragraphs: 18824
Final non-instruction paragraphs: 17479
Saved to: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\non_instruction_dataset\non_instruction_data.txt

Sample cleaned text:
 I am currently unable to access my JPMorgan Chase account due to verification methods that are not usable in my situation. SMS verification is inaccessible to me, and phone verification is unreliable because I am located outside the United States and calls frequently disconnect before the process can be completed.


In [4]:
# Optional verification cell (no extra file creation).

final_path = project_root / Path(config.non_instruction_dir) / config.non_instruction_filename
text = final_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

logger.info("Verification complete")
logger.info("Verified file: %s", final_path.resolve())
logger.info("Paragraphs: %s", len(paragraphs))
logger.info("Characters: %s", len(text))

print("Verified file:", final_path.resolve())
print("Paragraphs:", len(paragraphs))
print("Characters:", len(text))

2026-07-11 20:45:41,762 | INFO | non_instruction | Verification complete


2026-07-11 20:45:41,765 | INFO | non_instruction | Verified file: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\non_instruction_dataset\non_instruction_data.txt
2026-07-11 20:45:41,768 | INFO | non_instruction | Paragraphs: 17479
2026-07-11 20:45:41,768 | INFO | non_instruction | Characters: 7102844


Verified file: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\non_instruction_dataset\non_instruction_data.txt
Paragraphs: 17479
Characters: 7102844


In [5]:
# ============================================================
# Create Hugging Face Dataset
# ============================================================

from pathlib import Path
from datasets import Dataset, DatasetDict

non_instruction_dir = project_root / Path(config.non_instruction_dir)
non_instruction_dir.mkdir(parents=True, exist_ok=True)
input_path = non_instruction_dir / config.non_instruction_filename
dataset_dir = non_instruction_dir / config.hf_dataset_dirname

text = input_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

records = [{"text": p} for p in paragraphs]
full_ds = Dataset.from_list(records)

split_ds = full_ds.train_test_split(test_size=config.test_size, seed=config.seed)
hf_ds = DatasetDict(
    {
        "train": split_ds["train"],
        "validation": split_ds["test"],
    }
)

dataset_dir.mkdir(parents=True, exist_ok=True)
hf_ds.save_to_disk(str(dataset_dir))

logger.info("HF dataset creation complete")
logger.info("Total rows: %s", len(full_ds))
logger.info("Train rows: %s", len(hf_ds["train"]))
logger.info("Validation rows: %s", len(hf_ds["validation"]))
logger.info("Saved dataset to: %s", dataset_dir.resolve())

print("Total rows:", len(full_ds))
print("Train rows:", len(hf_ds["train"]))
print("Validation rows:", len(hf_ds["validation"]))
print("Saved to:", dataset_dir.resolve())
print("\nSample train record:\n", hf_ds["train"][0]["text"][:500])

d:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Saving the dataset (1/1 shards): 100%|██████████| 1748/1748 [00:00<00:00, 132805.19 examples/s]
2026-07-11 20:46:08,856 | INFO | non_instruction | HF dataset creation complete
2026-07-11 20:46:08,858 | INFO | non_instruction | Total rows: 17479
2026-07-11 20:46:08,859 | INFO | non_instruction | Train rows: 15731
2026-07-11 20:46:08,860 | INFO | non_instruction | Validation rows: 1748
2026-07-11 20:46:08,861 | INFO | non_instruction | Saved dataset to: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\non_instruction_dataset\hf_non_instruction_dataset


Total rows: 17479
Train rows: 15731
Validation rows: 1748
Saved to: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\non_instruction_dataset\hf_non_instruction_dataset

Sample train record:
 I would caution prospective customers ; especially those who value stability and predictability in their credit relationships, to consider this risk when choosing a card issuer.


In [6]:
print(hf_ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 15731
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1748
    })
})


In [7]:
# ============================================================
# 11. Load tokenizer
# ============================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)

# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: Qwen/Qwen2.5-0.5B
Vocab size: 151665
Pad token: <|endoftext|> | Pad token id: 151643
EOS token: <|endoftext|> | EOS token id: 151643


In [8]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================

dataset = hf_ds

def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"], truncation=True, max_length=config.block_size)

tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

tokenized_datasets
tokenized_datasets["train"]["input_ids"][0]

Tokenizing text corpus: 100%|██████████| 1748/1748 [00:01<00:00, 1746.79 examples/s]


[40,
 1035,
 27830,
 32447,
 6310,
 2587,
 5310,
 1846,
 879,
 897,
 19753,
 323,
 7023,
 2897,
 304,
 862,
 6668,
 11871,
 11,
 311,
 2908,
 419,
 5214,
 979,
 18774,
 264,
 3701,
 54835,
 13]

In [9]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = [block.copy() for block in input_id_blocks]

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [10]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

final_dataset
final_dataset["train"][0]

Creating fixed-size training blocks of 512 tokens: 100%|██████████| 15731/15731 [00:01<00:00, 8901.76 examples/s]
Creating fixed-size training blocks of 512 tokens: 100%|██████████| 1748/1748 [00:00<00:00, 8085.99 examples/s]


{'input_ids': [40,
  1035,
  27830,
  32447,
  6310,
  2587,
  5310,
  1846,
  879,
  897,
  19753,
  323,
  7023,
  2897,
  304,
  862,
  6668,
  11871,
  11,
  311,
  2908,
  419,
  5214,
  979,
  18774,
  264,
  3701,
  54835,
  13,
  1925,
  608,
  608,
  3157,
  1154,
  847,
  2692,
  572,
  11430,
  220,
  17,
  22,
  15,
  15,
  13,
  15,
  15,
  304,
  3633,
  448,
  264,
  7745,
  5435,
  311,
  11236,
  264,
  12425,
  1526,
  659,
  358,
  10602,
  311,
  5258,
  8160,
  369,
  279,
  1509,
  2587,
  4764,
  11,
  4518,
  315,
  12308,
  10514,
  11,
  220,
  17,
  22,
  15,
  15,
  13,
  15,
  15,
  572,
  49582,
  504,
  847,
  2692,
  2041,
  847,
  23715,
  13,
  7941,
  4588,
  369,
  847,
  4540,
  1372,
  11,
  2621,
  11,
  323,
  829,
  11,
  323,
  1221,
  28646,
  752,
  311,
  3725,
  847,
  9784,
  8234,
  5624,
  320,
  18180,
  45,
  7457,
  358,
  7069,
  28340,
  3170,
  458,
  18180,
  45,
  572,
  4362,
  11,
  36710,
  5538,
  4643,
  429,
  419,
  1035,


In [11]:
# ============================================================
# Check device availability
# ============================================================
import torch
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


In [12]:
# Clear memory before loading the model.
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()


In [13]:
from transformers import AutoModelForCausalLM

if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")
print("Model name:", config.model_name)
print("Training mode:", "QLoRA 4-bit" if use_cuda else "CPU full precision fallback")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:02<00:00, 111.84it/s]


Base model loaded successfully.
Model name: Qwen/Qwen2.5-0.5B
Training mode: CPU full precision fallback


In [14]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.
# This is cheaper than full fine-tuning and is widely used in real projects.
from peft import LoraConfig
from peft import TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [15]:
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)

In [16]:
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [17]:
# ============================================================
# 15. Data collator
# ============================================================
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [18]:
# ============================================================
# 16. Training arguments
# ============================================================
# These settings are designed for a small classroom/demo run.
# For larger corpora, increase dataset size, epochs, and evaluation frequency carefully.

import inspect
from transformers import TrainingArguments

In [19]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [20]:
from transformers import TrainingArguments
training_args = TrainingArguments(**training_kwargs)

In [21]:
# ============================================================
# 17. Build Trainer
# ============================================================
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

Trainer is ready.


In [22]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# 18. Start training
# ============================================================
train_result = trainer.train()
print("Training completed.")

Step,Training Loss


In [ ]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================
import os

trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

In [ ]:
# ============================================================
# 20. Push LoRA adapter to Hugging Face Hub
# ============================================================
repo_id = "sunny199/finance-non-instruction-lora"

# Make sure you are logged in first:
# from huggingface_hub import notebook_login
# notebook_login()

model.push_to_hub(
    repo_id,
    private=True
)

tokenizer.push_to_hub(
    repo_id,
    private=True
)

print(f"Pushed adapter and tokenizer to: {repo_id}")

In [ ]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================
# Clean old objects to free memory.

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

from transformers import AutoTokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

inference_model.eval()

print("Base model + LoRA adapter loaded successfully for inference.")

In [ ]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# ============================================================
# 23. Test text continuation
# ============================================================
prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

In [ ]:
# ============================================================
# 24. Optional merge step
# ============================================================
# This step merges the LoRA adapter into the base model.
# Use this only when you want a standalone model for deployment.

merged_model_dir = "/content/finance_qwen_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

# Reload the base model in float16 for safe merging.
merge_base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    merge_base_model,
    config.adapter_dir
)

# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

# Save the merged standalone model and tokenizer.
merged_model.save_pretrained(merged_model_dir)
inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")